# Phase 5 - Image Segmentation and Important Region Detection

Segment images using clustering and identify important regions by iteratively removing clusters based on their impact on classification confidence.

## Section 1: Import Libraries and Load Pre-trained Model

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CLASSES = [
    "butterfly", "cat", "chicken", "cow", "dog",
    "elephant", "horse", "sheep", "spider", "squirrel"
]

print("Loading pre-trained models from Phase 3...")
try:
    svm_model = joblib.load("svm_tuned_loguniform_scaled.pkl")
    print("Loaded: svm_tuned_loguniform_scaled.pkl")
except FileNotFoundError:
    try:
        svm_model = joblib.load("svm_phase3.pkl")
        print("Loaded: svm_phase3.pkl")
    except FileNotFoundError:
        raise FileNotFoundError("SVM model not found. Please run Phase 3 first.")

print("Models loaded successfully")
print("Note: SVM model is a Pipeline containing StandardScaler + SVC")

## Section 2: Feature Extraction Functions

In [ ]:
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import shannon_entropy
from scipy.stats import skew, kurtosis

PROCESS_SIZE = (128, 128)
GLCM_DISTANCES = [1]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]

def compute_rgb_stats(img):
    b, g, r = cv2.split(img.astype(np.float32))
    return r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std()

def compute_hsv_stats(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    return h.mean(), s.mean(), v.mean(), h.std(), s.std(), v.std()

def compute_intensity_stats(gray):
    g = gray.astype(np.float32)
    mean_int = g.mean()
    std_int = g.std()
    min_int = g.min()
    max_int = g.max()
    range_int = max_int - min_int
    flat = g.ravel()
    skewness_val = float(skew(flat, bias=False)) if flat.size > 1 else 0.0
    kurtosis_val = float(kurtosis(flat, fisher=True, bias=False)) if flat.size > 1 else 0.0
    entropy_val = float(shannon_entropy(gray))
    return mean_int, std_int, min_int, max_int, range_int, skewness_val, kurtosis_val, entropy_val

def compute_glcm_features(gray):
    gray_u8 = gray.astype(np.uint8)
    glcm = graycomatrix(gray_u8, distances=GLCM_DISTANCES, angles=GLCM_ANGLES, 
                        levels=256, symmetric=True, normed=True)
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
    vals = [float(graycoprops(glcm, p).mean()) for p in props]
    return vals

def compute_edge_features(gray):
    edges = cv2.Canny(gray, 100, 200)
    edge_density = float(np.count_nonzero(edges)) / edges.size
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    mask = edges > 0
    if np.any(mask):
        vals = mag[mask]
        edge_mean = float(vals.mean())
        edge_std = float(vals.std())
    else:
        edge_mean = 0.0
        edge_std = 0.0
    return edge_mean, edge_std, edge_density

def compute_contour_features(gray):
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0, 0.0, 0.0
    cnt = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(cnt))
    perimeter = float(cv2.arcLength(cnt, True))
    compactness = (perimeter ** 2) / (4 * np.pi * area) if area > 0 else 0.0
    return area, perimeter, compactness

def extract_image_features(img):
    img_resized = cv2.resize(img, PROCESS_SIZE)
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    h, w = img.shape[:2]
    width = float(w)
    height = float(h)
    aspect_ratio = width / height if height > 0 else 0.0
    
    mean_r, mean_g, mean_b, std_r, std_g, std_b = compute_rgb_stats(img_resized)
    mean_h, mean_s, mean_v, std_h, std_s, std_v = compute_hsv_stats(img_resized)
    brightness = mean_v
    contrast_simple = np.std(gray.astype(np.float32))
    
    mean_int, std_int, min_int, max_int, range_int, skewness_val, kurtosis_val, entropy_val = compute_intensity_stats(gray)
    contrast, dissimilarity, homogeneity, energy, correlation, ASM = compute_glcm_features(gray)
    edge_mean, edge_std, edge_density = compute_edge_features(gray)
    contour_area, contour_perimeter, compactness = compute_contour_features(gray)
    
    feats = np.array([
        width, height, aspect_ratio,
        mean_r, mean_g, mean_b, std_r, std_g, std_b,
        mean_h, mean_s, mean_v, std_h, std_s, std_v,
        brightness, contrast_simple,
        mean_int, std_int, min_int, max_int, range_int,
        skewness_val, kurtosis_val, entropy_val,
        contrast, dissimilarity, homogeneity, energy, correlation, ASM,
        edge_mean, edge_std, edge_density,
        contour_area, contour_perimeter, compactness
    ], dtype=np.float32)
    
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats

## Section 3: Image Segmentation Function

In [ ]:
def segment_image(image, n_clusters=5):
    h, w = image.shape[:2]
    
    pixel_features = []
    positions = []
    
    for y in range(h):
        for x in range(w):
            b, g, r = image[y, x]
            norm_x = x / w
            norm_y = y / h
            pixel_features.append([r, g, b, norm_x, norm_y])
            positions.append([y, x])
    
    pixel_features = np.array(pixel_features)
    positions = np.array(positions)
    
    scaler_pixel = StandardScaler()
    pixel_features_scaled = scaler_pixel.fit_transform(pixel_features)
    
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1000, n_init=3)
    labels = kmeans.fit_predict(pixel_features_scaled)
    
    segmented_image = labels.reshape(h, w)
    
    return segmented_image, labels, positions

## Section 4: Important Region Detection Function

In [ ]:
def find_important_clusters(image, segmented_image, labels, positions, true_label, n_final_clusters=3, model=None):
    if model is None:
        model = svm_model
    
    unique_clusters = np.unique(labels)
    n_clusters = len(unique_clusters)
    
    if n_clusters <= n_final_clusters:
        return unique_clusters
    
    class_idx = {}
    if hasattr(model, 'classes_'):
        class_idx = {cls: idx for idx, cls in enumerate(model.classes_)}
    
    original_features = extract_image_features(image)
    original_features = original_features.reshape(1, -1)
    
    if hasattr(model, 'predict_proba'):
        original_proba = model.predict_proba(original_features)[0]
        if true_label in class_idx:
            original_confidence = original_proba[class_idx[true_label]]
        else:
            original_confidence = np.max(original_proba)
    elif hasattr(model, 'decision_function'):
        original_decision = model.decision_function(original_features)[0]
        if true_label in class_idx:
            original_confidence = original_decision[class_idx[true_label]]
        else:
            original_confidence = np.max(original_decision)
    else:
        original_confidence = 1.0
    
    remaining_clusters = list(unique_clusters)
    
    while len(remaining_clusters) > n_final_clusters:
        min_impact = float('inf')
        cluster_to_remove = None
        
        for cluster_id in remaining_clusters:
            temp_image = np.zeros_like(image)
            
            for cluster in remaining_clusters:
                if cluster != cluster_id:
                    cluster_pos = positions[labels == cluster]
                    for pos in cluster_pos:
                        temp_image[pos[0], pos[1]] = image[pos[0], pos[1]]
            
            if np.sum(temp_image) == 0:
                continue
            
            temp_features = extract_image_features(temp_image)
            temp_features = temp_features.reshape(1, -1)
            
            if hasattr(model, 'predict_proba'):
                temp_proba = model.predict_proba(temp_features)[0]
                if true_label in class_idx:
                    temp_confidence = temp_proba[class_idx[true_label]]
                else:
                    temp_confidence = np.max(temp_proba)
            elif hasattr(model, 'decision_function'):
                temp_decision = model.decision_function(temp_features)[0]
                if true_label in class_idx:
                    temp_confidence = temp_decision[class_idx[true_label]]
                else:
                    temp_confidence = np.max(temp_decision)
            else:
                temp_confidence = 1.0
            
            impact = original_confidence - temp_confidence
            
            if impact < min_impact:
                min_impact = impact
                cluster_to_remove = cluster_id
        
        if cluster_to_remove is not None:
            remaining_clusters.remove(cluster_to_remove)
        else:
            break
    
    return np.array(remaining_clusters)

## Section 5: Process Dataset

In [ ]:
dataset_path = Path("Dataset/Animals-10")
output_path = Path("Dataset/Animals-10-Segmented")
output_path.mkdir(exist_ok=True)

N_CLUSTERS = 6
N_FINAL_CLUSTERS = 3

print(f"Processing dataset: {dataset_path}")
print(f"Initial clusters: {N_CLUSTERS}")
print(f"Final clusters: {N_FINAL_CLUSTERS}")
print(f"Output directory: {output_path}")

In [ ]:
results = []

for class_name in CLASSES:
    class_dir = dataset_path / class_name
    output_class_dir = output_path / class_name
    output_class_dir.mkdir(exist_ok=True)
    
    image_files = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
    
    print(f"\nProcessing class: {class_name} ({len(image_files)} images)")
    
    for img_path in tqdm(image_files, desc=class_name):
        try:
            image = cv2.imread(str(img_path))
            if image is None:
                continue
            
            image = cv2.resize(image, (224, 224))
            
            segmented_image, labels, positions = segment_image(image, n_clusters=N_CLUSTERS)
            
            important_clusters = find_important_clusters(
                image, segmented_image, labels, positions,
                true_label=class_name,
                n_final_clusters=N_FINAL_CLUSTERS,
                model=svm_model
            )
            
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            for cluster_id in important_clusters:
                cluster_positions = positions[labels == cluster_id]
                for pos in cluster_positions:
                    mask[pos[0], pos[1]] = 255
            
            segmented_result = cv2.bitwise_and(image, image, mask=mask)
            
            output_file = output_class_dir / img_path.name
            cv2.imwrite(str(output_file), segmented_result)
            
            results.append({
                'image': img_path.name,
                'class': class_name,
                'n_initial_clusters': N_CLUSTERS,
                'n_final_clusters': len(important_clusters),
                'removed_clusters': N_CLUSTERS - len(important_clusters)
            })
            
        except Exception as e:
            print(f"Error processing {img_path.name}: {str(e)}")
            continue

print("\nProcessing completed")

## Section 6: Save Results and Statistics

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("CSV/phase5_segmentation_results.csv", index=False)

print("\nSegmentation Statistics:")
print(f"Total images processed: {len(results_df)}")
print(f"\nCluster statistics per class:")
print(results_df.groupby('class')['n_final_clusters'].agg(['mean', 'min', 'max']))
print(f"\nAverage clusters removed: {results_df['removed_clusters'].mean():.2f}")

print(f"\nResults saved to CSV/phase5_segmentation_results.csv")
print(f"Segmented images saved to {output_path}")

## Section 7: Visualization Sample

In [ ]:
import matplotlib.pyplot as plt

sample_class = CLASSES[0]
original_dir = dataset_path / sample_class
segmented_dir = output_path / sample_class

sample_images = list(original_dir.glob("*.jpg"))[:3]

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle(f'Segmentation Results - {sample_class}', fontsize=16)

for idx, img_path in enumerate(sample_images):
    original = cv2.imread(str(img_path))
    original = cv2.resize(original, (224, 224))
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    
    segmented_path = segmented_dir / img_path.name
    segmented = cv2.imread(str(segmented_path))
    segmented_rgb = cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB)
    
    axes[idx, 0].imshow(original_rgb)
    axes[idx, 0].set_title('Original')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(segmented_rgb)
    axes[idx, 1].set_title('Important Regions')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()